In [ ]:
# Dropout Risk Management System - StudentForecastPortal by HNAIResearcher
# Predicts and monitors student dropout risk in real time using machine learning.

!pip install fastapi uvicorn scikit-learn pandas numpy pyngrok joblib matplotlib seaborn websockets requests python-multipart -q
print("Dropout Risk Management System - Environment Ready")


Dropout Risk Management System - Environment Ready


In [ ]:
# Step 2: Generate synthetic student dataset with 20 features
import pandas as pd
import numpy as np

np.random.seed(42)
n = 1000

data = {
    "attendance_rate":          np.random.uniform(20, 100, n),
    "assignment_completion":    np.random.uniform(10, 100, n),
    "quiz_avg_score":           np.random.uniform(20, 100, n),
    "previous_gpa":             np.random.uniform(0.5, 4.0, n),
    "failed_subjects":          np.random.randint(0, 6, n),
    "late_submissions":         np.random.uniform(0, 100, n),
    "lms_logins_per_week":      np.random.uniform(0, 30, n),
    "library_visits":           np.random.randint(0, 20, n),
    "office_hours_attended":    np.random.randint(0, 15, n),
    "study_group_participation":np.random.randint(0, 10, n),
    "financial_stress":         np.random.randint(1, 6, n),
    "commute_distance_km":      np.random.uniform(0, 100, n),
    "part_time_job":            np.random.randint(0, 2, n),
    "family_dependents":        np.random.randint(0, 8, n),
    "internet_access_quality":  np.random.randint(1, 6, n),
    "mental_health_score":      np.random.uniform(1, 10, n),
    "peer_relationship_score":  np.random.uniform(1, 10, n),
    "motivation_score":         np.random.uniform(1, 10, n),
    "first_generation_student": np.random.randint(0, 2, n),
    "gender":                   np.random.randint(0, 2, n),
}

df = pd.DataFrame(data)

df["dropout_risk"] = (
    (100 - df["attendance_rate"]) * 0.25 +
    (100 - df["assignment_completion"]) * 0.20 +
    (100 - df["quiz_avg_score"]) * 0.15 +
    (4.0 - df["previous_gpa"]) * 10 * 0.10 +
    df["failed_subjects"] * 5 * 0.10 +
    df["financial_stress"] * 5 * 0.08 +
    (10 - df["motivation_score"]) * 3 * 0.07 +
    (10 - df["mental_health_score"]) * 2 * 0.05
)

df["dropout_risk"] = (df["dropout_risk"] - df["dropout_risk"].min()) / \
                     (df["dropout_risk"].max() - df["dropout_risk"].min())

df["dropout_label"] = (df["dropout_risk"] > 0.5).astype(int)

print(f"Dataset shape: {df.shape}")
print(f"High risk students: {df['dropout_label'].sum()} ({df['dropout_label'].mean()*100:.1f}%)")
print(f"Low risk students: {(df['dropout_label']==0).sum()} ({(df['dropout_label']==0).mean()*100:.1f}%)")
print("\nSample data:")
print(df[["attendance_rate","assignment_completion","previous_gpa","dropout_risk","dropout_label"]].head(3))

Dataset shape: (1000, 22)
High risk students: 549 (54.9%)
Low risk students: 451 (45.1%)

Sample data:
   attendance_rate  assignment_completion  previous_gpa  dropout_risk  \
0        49.963210              26.661964      2.854460      0.735019   
1        96.057145              58.771085      3.288385      0.384161   
2        78.559515              88.565125      1.376638      0.214766   

   dropout_label  
0              1  
1              0  
2              0  


In [ ]:
# Step 3: Train 3 model versions — Logistic Regression, Random Forest, Gradient Boosting
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
import joblib
import os

# Features and target
FEATURES = [col for col in df.columns if col not in ["dropout_risk", "dropout_label"]]
X = df[FEATURES]
y = df["dropout_label"]

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save scaler
os.makedirs("models", exist_ok=True)
joblib.dump(scaler, "models/scaler.pkl")

# Define 3 model versions
model_configs = {
    "v1_logistic_regression": LogisticRegression(max_iter=1000, random_state=42),
    "v2_random_forest":       RandomForestClassifier(n_estimators=100, random_state=42),
    "v3_gradient_boosting":   GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# Train, evaluate, save each version
model_metadata = {}

for name, model in model_configs.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_prob = model.predict_proba(X_test_scaled)[:, 1]

    metadata = {
        "name": name,
        "accuracy": round(accuracy_score(y_test, y_pred), 4),
        "f1_score": round(f1_score(y_test, y_pred), 4),
        "auc_roc":  round(roc_auc_score(y_test, y_prob), 4),
    }
    model_metadata[name] = metadata
    joblib.dump(model, f"models/{name}.pkl")
    print(f" {name}: Accuracy={metadata['accuracy']} | F1={metadata['f1_score']} | AUC={metadata['auc_roc']}")

print("\nAll 3 models trained and saved ")
print(f"Features used: {len(FEATURES)}")

 v1_logistic_regression: Accuracy=0.975 | F1=0.9801 | AUC=0.9977
 v2_random_forest: Accuracy=0.905 | F1=0.9224 | AUC=0.9761
 v3_gradient_boosting: Accuracy=0.92 | F1=0.9355 | AUC=0.9789

All 3 models trained and saved 
Features used: 20


In [ ]:
# Step 4: Write the FastAPI backend to file
%%writefile app.py

# Dropout Risk Management System — StudentForecastPortal by HNAIResearcher
# FastAPI backend: serves predictions, model versions, metrics, and WebSocket stream

from fastapi import FastAPI, WebSocket, WebSocketDisconnect
from fastapi.middleware.cors import CORSMiddleware
import asyncio, joblib, numpy as np, time, os, random
from datetime import datetime

app = FastAPI(title="Dropout Risk Management System")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Load all models and scaler on startup
FEATURES = [
    "attendance_rate","assignment_completion","quiz_avg_score","previous_gpa",
    "failed_subjects","late_submissions","lms_logins_per_week","library_visits",
    "office_hours_attended","study_group_participation","financial_stress",
    "commute_distance_km","part_time_job","family_dependents","internet_access_quality",
    "mental_health_score","peer_relationship_score","motivation_score",
    "first_generation_student","gender"
]

scaler  = joblib.load("models/scaler.pkl")
models  = {
    "v1_logistic_regression": joblib.load("models/v1_logistic_regression.pkl"),
    "v2_random_forest":       joblib.load("models/v2_random_forest.pkl"),
    "v3_gradient_boosting":   joblib.load("models/v3_gradient_boosting.pkl"),
}
model_metadata = {
    "v1_logistic_regression": {"accuracy":0.975,"f1_score":0.9801,"auc_roc":0.9977,"algorithm":"Logistic Regression"},
    "v2_random_forest":       {"accuracy":0.905,"f1_score":0.9224,"auc_roc":0.9761,"algorithm":"Random Forest"},
    "v3_gradient_boosting":   {"accuracy":0.920,"f1_score":0.9355,"auc_roc":0.9789,"algorithm":"Gradient Boosting"},
}

# System state
state = {
    "active_model": "v3_gradient_boosting",
    "sensitivity":  0.5,
}

# Pakistani student names
STUDENT_NAMES = [
    "Ahmed Raza","Fatima Malik","Usman Ali","Ayesha Khan","Bilal Ahmed",
    "Zara Hussain","Omar Farooq","Sana Iqbal","Hassan Nawaz","Nida Jameel",
    "Tariq Mehmood","Amna Sheikh","Saad Butt","Hira Baig","Faisal Chaudhry"
]

def generate_student():
    return {
        "name": random.choice(STUDENT_NAMES),
        "attendance_rate":          random.uniform(20, 100),
        "assignment_completion":    random.uniform(10, 100),
        "quiz_avg_score":           random.uniform(20, 100),
        "previous_gpa":             random.uniform(0.5, 4.0),
        "failed_subjects":          random.randint(0, 5),
        "late_submissions":         random.uniform(0, 100),
        "lms_logins_per_week":      random.uniform(0, 30),
        "library_visits":           random.randint(0, 20),
        "office_hours_attended":    random.randint(0, 15),
        "study_group_participation":random.randint(0, 10),
        "financial_stress":         random.randint(1, 5),
        "commute_distance_km":      random.uniform(0, 100),
        "part_time_job":            random.randint(0, 1),
        "family_dependents":        random.randint(0, 7),
        "internet_access_quality":  random.randint(1, 5),
        "mental_health_score":      random.uniform(1, 10),
        "peer_relationship_score":  random.uniform(1, 10),
        "motivation_score":         random.uniform(1, 10),
        "first_generation_student": random.randint(0, 1),
        "gender":                   random.randint(0, 1),
    }

def predict_risk(student_data):
    features = [student_data[f] for f in FEATURES]
    scaled   = scaler.transform([features])
    model    = models[state["active_model"]]
    start    = time.time()
    prob     = model.predict_proba(scaled)[0][1]
    latency  = round((time.time() - start) * 1000, 2)
    risk     = prob
    label    = "high" if risk > state["sensitivity"] else "low" if risk < 0.3 else "moderate"
    intervention = (
        "Schedule urgent academic counseling session" if label == "high"
        else "Send motivational check-in message" if label == "moderate"
        else "Student is on track — continue monitoring"
    )
    return {
        "risk_score":    round(risk, 4),
        "risk_label":    label,
        "intervention":  intervention,
        "latency_ms":    latency,
        "model_used":    state["active_model"],
        "timestamp":     datetime.now().isoformat(),
    }

# --- REST Endpoints ---

@app.get("/")
def root():
    return {"system": "Dropout Risk Management System", "status": "running"}

@app.get("/versions")
def get_versions():
    result = []
    for k, v in model_metadata.items():
        result.append({
            "id": k,
            "algorithm": v["algorithm"],
            "accuracy":  v["accuracy"],
            "f1_score":  v["f1_score"],
            "auc_roc":   v["auc_roc"],
            "active":    k == state["active_model"]
        })
    return {"versions": result}

@app.post("/versions/switch/{model_id}")
def switch_model(model_id: str):
    if model_id not in models:
        return {"error": "Model not found"}
    state["active_model"] = model_id
    return {"message": f"Switched to {model_id}", "active_model": model_id}

@app.post("/sensitivity/{value}")
def set_sensitivity(value: float):
    state["sensitivity"] = max(0.1, min(0.9, value))
    return {"sensitivity": state["sensitivity"]}

@app.get("/metrics")
def get_metrics():
    active = state["active_model"]
    meta   = model_metadata[active]
    return {
        "active_model": active,
        "accuracy":     meta["accuracy"],
        "f1_score":     meta["f1_score"],
        "auc_roc":      meta["auc_roc"],
        "sensitivity":  state["sensitivity"],
        "total_models": len(models),
    }

@app.post("/predict")
def predict(student: dict):
    return predict_risk(student)

# --- WebSocket Endpoint ---

@app.websocket("/ws")
async def websocket_endpoint(websocket: WebSocket):
    await websocket.accept()
    try:
        while True:
            student = generate_student()
            result  = predict_risk(student)
            result["student_name"] = student["name"]
            result["features"] = {
                "attendance":   round(student["attendance_rate"], 1),
                "assignments":  round(student["assignment_completion"], 1),
                "quiz_score":   round(student["quiz_avg_score"], 1),
                "gpa":          round(student["previous_gpa"], 2),
                "motivation":   round(student["motivation_score"], 1),
                "mental_health":round(student["mental_health_score"], 1),
            }
            await websocket.send_json(result)
            await asyncio.sleep(1.5)
    except WebSocketDisconnect:
        pass

Overwriting app.py


In [ ]:
# Step 5: Start the FastAPI server
import subprocess
import time

server_process = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(3)

if server_process.poll() is None:
    print("Server is running ")
else:
    print("Server crashed. Here's why:")
    print(server_process.stdout.read())

Server crashed. Here's why:
INFO:     Started server process [10548]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.



In [ ]:
# Step 6: Open ngrok tunnel
from pyngrok import ngrok

ngrok.set_auth_token("3I0jBXpRNWAolOiASDNmSjqmKA0_2BSJv9zYz5NhH54fEBTgi")

public_url = ngrok.connect(8000, "http")
print(f"Public URL: {public_url}")

Public URL: NgrokTunnel: "https://carport-curtain-freckles.ngrok-free.dev" -> "http://localhost:8000"


In [35]:
# Step 7: Test all API endpoints
import requests

BASE = "https://carport-curtain-freckles.ngrok-free.dev"

# Test root
r = requests.get(f"{BASE}/", headers={"ngrok-skip-browser-warning": "true"})
print("Root:", r.json())

# Test versions
r = requests.get(f"{BASE}/versions", headers={"ngrok-skip-browser-warning": "true"})
print("\nVersions:")
for v in r.json()["versions"]:
    print(f"  {v['id']} | Accuracy: {v['accuracy']} | Active: {v['active']}")

# Test metrics
r = requests.get(f"{BASE}/metrics", headers={"ngrok-skip-browser-warning": "true"})
print("\nMetrics:", r.json())

# Test sensitivity
r = requests.post(f"{BASE}/sensitivity/0.6", headers={"ngrok-skip-browser-warning": "true"})
print("\nSensitivity:", r.json())

# Test model switch
r = requests.post(f"{BASE}/versions/switch/v2_random_forest", headers={"ngrok-skip-browser-warning": "true"})
print("\nModel switch:", r.json())

Root: {'system': 'Dropout Risk Management System', 'status': 'running'}

Versions:
  v1_logistic_regression | Accuracy: 0.975 | Active: False
  v2_random_forest | Accuracy: 0.905 | Active: False
  v3_gradient_boosting | Accuracy: 0.92 | Active: True

Metrics: {'active_model': 'v3_gradient_boosting', 'accuracy': 0.92, 'f1_score': 0.9355, 'auc_roc': 0.9789, 'sensitivity': 0.5, 'total_models': 3}

Sensitivity: {'sensitivity': 0.6}

Model switch: {'message': 'Switched to v2_random_forest', 'active_model': 'v2_random_forest'}


In [ ]:
%%writefile dashboard.html
<!DOCTYPE html>
<html>
<head>
<title>Dropout Risk Management System</title>
<style>
* { box-sizing: border-box; margin: 0; padding: 0; }
body { font-family: 'Segoe UI', sans-serif; background: #0f1117; color: #eee; }

/* Header */
.header { background: #1a1d2e; padding: 16px 24px; display: flex; align-items: center; justify-content: space-between; border-bottom: 1px solid #2a2d3e; }
.header h1 { font-size: 18px; font-weight: 600; color: #fff; }
.header .subtitle { font-size: 12px; color: #888; margin-top: 2px; }
.header .brand { font-size: 12px; color: #4ade80; }
.status-dot { width: 8px; height: 8px; border-radius: 50%; background: #4ade80; display: inline-block; margin-right: 6px; animation: pulse 1.5s infinite; }
@keyframes pulse { 0%,100%{opacity:1} 50%{opacity:0.4} }

/* Tabs */
.tabs { display: flex; background: #1a1d2e; border-bottom: 1px solid #2a2d3e; }
.tab { padding: 12px 24px; font-size: 13px; cursor: pointer; color: #888; border-bottom: 2px solid transparent; transition: all 0.2s; }
.tab.active { color: #4ade80; border-bottom-color: #4ade80; }
.tab:hover { color: #fff; }

/* Panels */
.panel { display: none; padding: 20px; }
.panel.active { display: block; }

/* Cards */
.card { background: #1a1d2e; border-radius: 12px; padding: 16px; border: 1px solid #2a2d3e; }
.card-title { font-size: 12px; color: #888; margin-bottom: 12px; text-transform: uppercase; letter-spacing: 0.5px; }
.grid-3 { display: grid; grid-template-columns: repeat(3,1fr); gap: 12px; margin-bottom: 16px; }
.grid-2 { display: grid; grid-template-columns: repeat(2,1fr); gap: 12px; margin-bottom: 16px; }

/* Risk gauge */
.gauge-wrap { display: flex; flex-direction: column; align-items: center; }
.gauge-num { font-size: 32px; font-weight: 600; margin-top: 8px; }
.gauge-label { font-size: 13px; color: #888; margin-top: 4px; }

/* Stat cards */
.stat { background: #1a1d2e; border-radius: 10px; padding: 14px; border: 1px solid #2a2d3e; }
.stat-label { font-size: 11px; color: #888; margin-bottom: 6px; }
.stat-value { font-size: 20px; font-weight: 500; }

/* Alerts */
.alert-item { padding: 10px 14px; border-radius: 8px; font-size: 13px; margin-bottom: 8px; }
.alert-high { background: rgba(239,68,68,0.15); color: #f87171; border-left: 3px solid #ef4444; }
.alert-moderate { background: rgba(234,179,8,0.15); color: #facc15; border-left: 3px solid #eab308; }
.alert-low { background: rgba(74,222,128,0.15); color: #4ade80; border-left: 3px solid #22c55e; }

/* Factor bars */
.factor-row { display: flex; align-items: center; gap: 10px; margin-bottom: 10px; }
.factor-name { font-size: 12px; color: #888; min-width: 80px; }
.factor-bar-wrap { flex: 1; height: 6px; background: #2a2d3e; border-radius: 3px; overflow: hidden; }
.factor-bar { height: 100%; border-radius: 3px; transition: width 0.4s; }
.factor-val { font-size: 12px; color: #888; min-width: 35px; text-align: right; }

/* Pipeline */
.pipeline { display: flex; align-items: center; gap: 8px; margin-bottom: 16px; flex-wrap: wrap; }
.stage { padding: 8px 14px; border-radius: 8px; font-size: 12px; background: #2a2d3e; color: #888; transition: all 0.3s; }
.stage.active { background: rgba(74,222,128,0.2); color: #4ade80; }
.stage.done { background: rgba(74,222,128,0.1); color: #4ade80; }
.arrow { color: #444; font-size: 14px; }

/* Model version cards */
.version-card { background: #1a1d2e; border-radius: 10px; padding: 14px; border: 2px solid #2a2d3e; margin-bottom: 10px; display: flex; align-items: center; justify-content: space-between; cursor: pointer; transition: all 0.2s; }
.version-card.active-model { border-color: #4ade80; }
.version-card:hover { border-color: #4ade80; }
.version-btn { padding: 6px 14px; border-radius: 6px; font-size: 12px; border: none; cursor: pointer; background: #4ade80; color: #0f1117; font-weight: 500; }

/* Slider */
.slider-wrap { display: flex; align-items: center; gap: 12px; margin-bottom: 16px; }
.slider-wrap label { font-size: 13px; color: #888; min-width: 120px; }
input[type=range] { flex: 1; accent-color: #4ade80; }
.slider-val { font-size: 13px; font-weight: 500; min-width: 35px; }

/* Feed */
.feed-item { background: #1a1d2e; border-radius: 8px; padding: 12px 14px; margin-bottom: 8px; border: 1px solid #2a2d3e; display: flex; justify-content: space-between; align-items: center; }
.feed-name { font-size: 13px; font-weight: 500; }
.feed-sub { font-size: 11px; color: #888; margin-top: 2px; }
.risk-badge { padding: 4px 10px; border-radius: 20px; font-size: 11px; font-weight: 500; }
.badge-high { background: rgba(239,68,68,0.2); color: #f87171; }
.badge-moderate { background: rgba(234,179,8,0.2); color: #facc15; }
.badge-low { background: rgba(74,222,128,0.2); color: #4ade80; }

button { cursor: pointer; }
</style>
</head>
<body>

<div class="header">
  <div>
    <h1>Dropout Risk Management System</h1>
    <div class="subtitle">StudentForecastPortal · by HNAIResearcher</div>
  </div>
  <div style="display:flex; align-items:center; gap:16px;">
    <span style="font-size:13px;"><span class="status-dot"></span><span id="connStatus">Connecting...</span></span>
    <span class="brand">HNAIResearcher</span>
  </div>
</div>

<div class="tabs">
  <div class="tab active" onclick="switchTab('teacher')">🎓 Teacher View</div>
  <div class="tab" onclick="switchTab('admin')">⚙️ Admin View</div>
  <div class="tab" onclick="switchTab('student')">👤 Student View</div>
  <div class="tab" onclick="switchTab('pipeline')">🔄 Pipeline</div>
</div>

<!-- TEACHER VIEW -->
<div id="tab-teacher" class="panel active">
  <div class="grid-3">
    <div class="stat"><div class="stat-label">Students monitored</div><div class="stat-value" id="totalCount">0</div></div>
    <div class="stat"><div class="stat-label">High risk alerts</div><div class="stat-value" id="highCount" style="color:#f87171;">0</div></div>
    <div class="stat"><div class="stat-label">Active model</div><div class="stat-value" style="font-size:13px;" id="activeModelStat">--</div></div>
  </div>

  <div class="grid-2">
    <div class="card">
      <div class="card-title">Live risk gauge</div>
      <div class="gauge-wrap">
        <svg viewBox="0 0 160 90" style="width:180px;">
          <path d="M20 85 A60 60 0 0 1 140 85" fill="none" stroke="#2a2d3e" stroke-width="14" stroke-linecap="round"></path>
          <path id="gaugeArc" d="M20 85 A60 60 0 0 1 140 85" fill="none" stroke="#4ade80" stroke-width="14" stroke-linecap="round" stroke-dasharray="188" stroke-dashoffset="188"></path>
        </svg>
        <div class="gauge-num" id="gaugeNum">--</div>
        <div class="gauge-label" id="gaugeLabel">waiting for data</div>
      </div>
    </div>

    <div class="card">
      <div class="card-title">Top risk factors</div>
      <div id="factorBars"></div>
    </div>
  </div>

  <div class="card" style="margin-bottom:16px;">
    <div class="card-title">Live student feed</div>
    <div id="studentFeed"></div>
  </div>

  <div class="card">
    <div class="card-title">Alerts & interventions</div>
    <div id="alertFeed"></div>
  </div>
</div>

<!-- ADMIN VIEW -->
<div id="tab-admin" class="panel">
  <div class="card" style="margin-bottom:16px;">
    <div class="card-title">Model versions — click to switch active model</div>
    <div id="versionCards"></div>
  </div>

  <div class="card" style="margin-bottom:16px;">
    <div class="card-title">Sensitivity threshold</div>
    <div class="slider-wrap">
      <label>Risk threshold</label>
      <input type="range" id="sensitivitySlider" min="0.1" max="0.9" step="0.05" value="0.5">
      <span class="slider-val" id="sensitivityVal">0.5</span>
    </div>
    <p style="font-size:12px; color:#888;">Higher = more sensitive (flags more students as at-risk)</p>
  </div>

  <div class="grid-3">
    <div class="stat"><div class="stat-label">Active model accuracy</div><div class="stat-value" id="metricAccuracy">--</div></div>
    <div class="stat"><div class="stat-label">F1 score</div><div class="stat-value" id="metricF1">--</div></div>
    <div class="stat"><div class="stat-label">AUC-ROC</div><div class="stat-value" id="metricAUC">--</div></div>
  </div>
</div>

<!-- STUDENT VIEW -->
<div id="tab-student" class="panel">
  <div class="card" style="margin-bottom:16px; text-align:center; padding:24px;">
    <div style="font-size:14px; color:#888; margin-bottom:8px;">Your current risk level</div>
    <div class="gauge-num" id="studentRisk" style="font-size:48px;">--</div>
    <div id="studentLabel" style="font-size:14px; margin-top:8px; color:#888;">--</div>
    <div id="studentTrend" style="font-size:13px; margin-top:12px; color:#4ade80;"></div>
  </div>

  <div class="card" style="margin-bottom:16px;">
    <div class="card-title">What's affecting your score</div>
    <div id="studentFactors"></div>
  </div>

  <div class="card">
    <div class="card-title">Recommended action</div>
    <div id="studentIntervention" style="font-size:14px; padding:8px 0; color:#eee;"></div>
  </div>
</div>

<!-- PIPELINE VIEW -->
<div id="tab-pipeline" class="panel">
  <div class="card" style="margin-bottom:16px;">
    <div class="card-title">Live prediction pipeline</div>
    <div class="pipeline">
      <div class="stage" id="stage0">Data input</div>
      <div class="arrow">→</div>
      <div class="stage" id="stage1">Validation</div>
      <div class="arrow">→</div>
      <div class="stage" id="stage2">Feature scaling</div>
      <div class="arrow">→</div>
      <div class="stage" id="stage3">Risk scoring</div>
      <div class="arrow">→</div>
      <div class="stage" id="stage4">Intervention</div>
    </div>
    <div id="pipelineStatus" style="font-size:13px; color:#888; margin-top:8px;">Waiting for prediction...</div>
  </div>

  <div class="card" style="margin-bottom:16px;">
    <div class="card-title">Last prediction details</div>
    <div id="pipelineDetails" style="font-size:13px; color:#888;">No prediction yet</div>
  </div>

  <div class="card">
    <div class="card-title">Model performance leaderboard</div>
    <div id="leaderboard"></div>
  </div>
</div>

<script>
const BASE = "https://carport-curtain-freckles.ngrok-free.dev";
const WS   = BASE.replace("https://","wss://") + "/ws";
const HDR  = {"ngrok-skip-browser-warning":"true"};

let totalCount = 0, highCount = 0;
let prevRisk = null;
let lastData = null;

// Tab switching
function switchTab(name) {
  document.querySelectorAll(".tab").forEach((t,i) => t.classList.remove("active"));
  document.querySelectorAll(".panel").forEach(p => p.classList.remove("active"));
  const tabs = ["teacher","admin","student","pipeline"];
  document.querySelectorAll(".tab")[tabs.indexOf(name)].classList.add("active");
  document.getElementById("tab-"+name).classList.add("active");
}

// Color helpers
function colorFor(label) {
  if (label==="high") return "#f87171";
  if (label==="moderate") return "#facc15";
  return "#4ade80";
}
function badgeClass(label) {
  return label==="high" ? "badge-high" : label==="moderate" ? "badge-moderate" : "badge-low";
}

// Update gauge
function updateGauge(risk, label) {
  const pct = risk * 100;
  const offset = 188 - (risk * 188);
  const arc = document.getElementById("gaugeArc");
  arc.style.strokeDashoffset = offset;
  arc.style.stroke = colorFor(label);
  document.getElementById("gaugeNum").textContent = Math.round(pct) + "%";
  document.getElementById("gaugeNum").style.color = colorFor(label);
  document.getElementById("gaugeLabel").textContent = label + " risk";
}

// Update factor bars
function updateFactors(features, label, containerId) {
  const el = document.getElementById(containerId);
  const items = [
    {name:"attendance", val:features.attendance, max:100},
    {name:"assignments", val:features.assignments, max:100},
    {name:"quiz score", val:features.quiz_score, max:100},
    {name:"GPA", val:features.gpa*25, max:100},
    {name:"motivation", val:features.motivation*10, max:100},
  ];
  el.innerHTML = items.map(f => `
    <div class="factor-row">
      <span class="factor-name">${f.name}</span>
      <div class="factor-bar-wrap">
        <div class="factor-bar" style="width:${f.val.toFixed(0)}%; background:${colorFor(label)};"></div>
      </div>
      <span class="factor-val">${f.val.toFixed(0)}%</span>
    </div>`).join("");
}

// Update student feed
function updateFeed(data) {
  const feed = document.getElementById("studentFeed");
  const item = document.createElement("div");
  item.className = "feed-item";
  item.innerHTML = `
    <div>
      <div class="feed-name">${data.student_name}</div>
      <div class="feed-sub">Attendance: ${data.features.attendance}% · GPA: ${data.features.gpa} · Latency: ${data.latency_ms}ms</div>
    </div>
    <span class="risk-badge ${badgeClass(data.risk_label)}">${data.risk_label} · ${Math.round(data.risk_score*100)}%</span>`;
  feed.prepend(item);
  if (feed.children.length > 5) feed.removeChild(feed.lastChild);
}

// Update alerts
function updateAlerts(data) {
  if (data.risk_label === "low") return;
  const feed = document.getElementById("alertFeed");
  const item = document.createElement("div");
  item.className = `alert-item alert-${data.risk_label}`;
  item.innerHTML = `<strong>${data.student_name}</strong> — ${data.intervention}`;
  feed.prepend(item);
  if (feed.children.length > 5) feed.removeChild(feed.lastChild);
}

// Pipeline animation
async function animatePipeline(data) {
  const stages = ["stage0","stage1","stage2","stage3","stage4"];
  const labels = ["Data input","Validation","Feature scaling","Risk scoring","Intervention"];
  for (let i = 0; i < stages.length; i++) {
    stages.forEach(s => document.getElementById(s).className = "stage done");
    document.getElementById(stages[i]).className = "stage active";
    document.getElementById("pipelineStatus").textContent = `Processing: ${labels[i]}...`;
    await new Promise(r => setTimeout(r, 200));
  }
  stages.forEach(s => document.getElementById(s).className = "stage done");
  document.getElementById("pipelineStatus").textContent =
    `✅ Complete — ${data.student_name} scored ${Math.round(data.risk_score*100)}% risk in ${data.latency_ms}ms`;
  document.getElementById("pipelineDetails").innerHTML = `
    <strong>Student:</strong> ${data.student_name}<br>
    <strong>Risk score:</strong> ${Math.round(data.risk_score*100)}% (${data.risk_label})<br>
    <strong>Model used:</strong> ${data.model_used}<br>
    <strong>Intervention:</strong> ${data.intervention}<br>
    <strong>Prediction latency:</strong> ${data.latency_ms}ms<br>
    <strong>Timestamp:</strong> ${data.timestamp}`;
}

// Load versions into admin panel
async function loadVersions() {
  const r = await fetch(`${BASE}/versions`, {headers: HDR});
  const data = await r.json();
  const el = document.getElementById("versionCards");
  el.innerHTML = data.versions.map(v => `
    <div class="version-card ${v.active ? 'active-model' : ''}" id="vc-${v.id}">
      <div>
        <div style="font-size:14px; font-weight:500;">${v.algorithm}</div>
        <div style="font-size:12px; color:#888; margin-top:4px;">
          Accuracy: ${(v.accuracy*100).toFixed(1)}% · F1: ${v.f1_score} · AUC: ${v.auc_roc}
        </div>
      </div>
      <button class="version-btn" onclick="switchModel('${v.id}')">
        ${v.active ? "✅ Active" : "Switch"}
      </button>
    </div>`).join("");

  // Update leaderboard
  const lb = document.getElementById("leaderboard");
  const sorted = [...data.versions].sort((a,b) => b.accuracy - a.accuracy);
  lb.innerHTML = sorted.map((v,i) => `
    <div style="display:flex; justify-content:space-between; align-items:center; padding:10px 0; border-bottom:1px solid #2a2d3e;">
      <div>
        <span style="font-size:13px; color:#888; margin-right:8px;">#${i+1}</span>
        <span style="font-size:13px; font-weight:500;">${v.algorithm}</span>
      </div>
      <div style="font-size:12px; color:#888;">
        Acc: ${(v.accuracy*100).toFixed(1)}% · F1: ${v.f1_score} · AUC: ${v.auc_roc}
      </div>
    </div>`).join("");
}

// Switch active model
async function switchModel(id) {
  await fetch(`${BASE}/versions/switch/${id}`, {method:"POST", headers:HDR});
  await loadVersions();
  await loadMetrics();
}

// Load metrics
async function loadMetrics() {
  const r = await fetch(`${BASE}/metrics`, {headers:HDR});
  const m = await r.json();
  document.getElementById("metricAccuracy").textContent = (m.accuracy*100).toFixed(1)+"%";
  document.getElementById("metricF1").textContent = m.f1_score;
  document.getElementById("metricAUC").textContent = m.auc_roc;
  document.getElementById("activeModelStat").textContent = m.active_model.replace("v1_","").replace("v2_","").replace("v3_","").replace("_"," ");
}

// Sensitivity slider
document.getElementById("sensitivitySlider").addEventListener("change", async (e) => {
  const val = e.target.value;
  document.getElementById("sensitivityVal").textContent = val;
  await fetch(`${BASE}/sensitivity/${val}`, {method:"POST", headers:HDR});
});

// WebSocket connection
function connect() {
  const ws = new WebSocket(WS);

  ws.onopen = () => {
    document.getElementById("connStatus").textContent = "Live";
  };

  ws.onclose = () => {
    document.getElementById("connStatus").textContent = "Reconnecting...";
    setTimeout(connect, 2000);
  };

  ws.onmessage = (event) => {
    const data = JSON.parse(event.data);
    lastData = data;

    // Update counts
    totalCount++;
    if (data.risk_label === "high") highCount++;
    document.getElementById("totalCount").textContent = totalCount;
    document.getElementById("highCount").textContent = highCount;

    // Teacher view
    updateGauge(data.risk_score, data.risk_label);
    updateFactors(data.features, data.risk_label, "factorBars");
    updateFeed(data);
    updateAlerts(data);

    // Student view
    const pct = Math.round(data.risk_score * 100);
    document.getElementById("studentRisk").textContent = pct + "%";
    document.getElementById("studentRisk").style.color = colorFor(data.risk_label);
    document.getElementById("studentLabel").textContent = data.risk_label + " risk";
    if (prevRisk !== null) {
      const diff = pct - Math.round(prevRisk * 100);
      document.getElementById("studentTrend").textContent =
        diff < 0 ? `↓ Improved by ${Math.abs(diff)}% — keep it up!` :
        diff > 0 ? `↑ Risk increased by ${diff}% — take action` :
        "→ Holding steady";
    }
    prevRisk = data.risk_score;
    updateFactors(data.features, data.risk_label, "studentFactors");
    document.getElementById("studentIntervention").textContent = data.intervention;

    // Pipeline view
    animatePipeline(data);
  };
}

// Initialize
loadVersions();
loadMetrics();
connect();
</script>
</body>
</html>

Overwriting dashboard.html


In [ ]:
# Download Dashboard

from google.colab import files
files.download("dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [34]:
# Kill everything on port 8000 and restart clean
import subprocess
import time

# Kill whatever is holding port 8000
subprocess.run(["fuser", "-k", "8000/tcp"], capture_output=True)
time.sleep(2)

# Start fresh
server_process = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)
time.sleep(3)

if server_process.poll() is None:
    print("Server is running ")
else:
    print("Crashed:", server_process.stdout.read())

Server is running 


In [31]:
%%writefile README.md
# Dropout Risk Management System
## StudentForecastPortal by HNAIResearcher

### What it does
A real-time student dropout risk prediction and early intervention system built for Pakistani university context. The system continuously monitors student data across 20 features — academic, behavioral, socioeconomic, and psychological — and streams live risk predictions through a WebSocket-powered dashboard with three role-based views.

### Five reference patterns covered
1. **Live streaming + sensitivity slider** — WebSocket streams risk scores continuously, threshold slider adjusts sensitivity live via API
2. **Model versioning & rollback** — 3 trained model versions (Logistic Regression, Random Forest, Gradient Boosting) with live switching and A/B comparison
3. **Metrics endpoint** — /metrics reports accuracy, F1, AUC-ROC, and prediction latency per request
4. **Multi-stage pipeline** — 5-stage visual pipeline (Data input → Validation → Feature scaling → Risk scoring → Intervention) with per-stage animation
5. **Model leaderboard** — ranked comparison of all 3 versions by accuracy, F1, and AUC

### Three user views
- **Teacher view** — live class risk gauge, student feed, alerts with intervention suggestions
- **Admin view** — model version panel, sensitivity control, performance metrics
- **Student view** — personal risk score, trend arrow, what's affecting score, recommended action

### Tech stack
- **Model:** scikit-learn (Logistic Regression, Random Forest, Gradient Boosting)
- **Backend:** FastAPI + WebSocket + Uvicorn
- **Tunnel:** ngrok (browser access from Colab)
- **Frontend:** Vanilla HTML/CSS/JavaScript
- **Data:** Synthetic Pakistani university student dataset (1000 students, 20 features)

### How to run
1. Open `Dropout Risk Management System.ipynb` in Google Colab
2. Run all cells in order (1 through 6)
3. Copy the ngrok URL printed in Cell 6
4. Update `BASE` in `dashboard.html` with the new URL
5. Download and open `dashboard.html` in your browser
6. Visit the ngrok URL once to clear the browser warning, then refresh the dashboard

### Model performance
| Model | Accuracy | F1 | AUC-ROC |
|---|---|---|---|
| Logistic Regression | 97.5% | 0.9801 | 0.9977 |
| Gradient Boosting | 92.0% | 0.9355 | 0.9789 |
| Random Forest | 90.5% | 0.9224 | 0.9761 |

### By
Humera Noor Ahmad — AI Researcher & Educator
HNAIResearcher | Faisalabad, Pakistan

Writing README.md


In [33]:
# Download all submission files
from google.colab import files

files.download("README.md")
files.download("app.py")
files.download("dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>